# Global Direct Multi-Horizon GBDT Experiments
Pooled CatBoost and LightGBM models. Both predict `log1p(OrderVolume)` and receive no actual future AppSessions.

In [ ]:
from pathlib import Path
import sys, pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.eda_pipeline import Paths
from src.model_pipeline import walk_forward_models
paths = Paths(ROOT)

## Build the rolling-origin training matrix
Run once from a terminal before full experiments: `python -m src.eda_pipeline build-training-origins`. Complete 42-day blocks are stored as Parquet; blocks affected by the known missing panel period are excluded.

Run feature groups in order. Begin with `static`, followed by `orders`, `orders_sessions`, `history`, and finally `all`. For each validation fold, only origin tables whose target window ends at or before that validation origin are eligible for training.

In [ ]:
FEATURE_GROUP = 'orders_sessions'
BACKENDS = ('catboost', 'lightgbm')
FOLDS = (1, 2, 3, 4)
metrics, oof = walk_forward_models(paths, FEATURE_GROUP, backends=BACKENDS, folds_to_run=FOLDS)
metrics.query("horizon_bucket == 'all'").groupby('model').rmsle.agg(['mean','std']).sort_values('mean')

In [ ]:
metrics[~metrics.horizon_bucket.isin(['all','open_only'])].groupby(['model','horizon_bucket']).rmsle.mean().unstack(0)